# 04 - Aprendizaje Supervisado (Students, columnas en espanol)

Objetivo: predecir `alguna_vez_probation`.

In [ ]:
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'reports').exists():
            return p
    raise RuntimeError('No se encontro la raiz del proyecto')

ROOT = find_project_root(Path.cwd().resolve())
print('Project root:', ROOT)

In [ ]:
path = ROOT / 'data/processed/students/students_limpio.csv'
out_model = ROOT / 'data/processed/students/students_modelado.csv'
fig_cm = ROOT / 'reports/students/figures/05_matriz_confusion_students.png'
metrics_path = ROOT / 'reports/students/metricas_students.json'

df = pd.read_csv(path)
df['alguna_vez_probation'] = df['alguna_vez_probation'].map({'No': 0, 'Yes': 1})
df = df.dropna(subset=['alguna_vez_probation']).copy()
df['alguna_vez_probation'] = df['alguna_vez_probation'].astype(int)

df.to_csv(out_model, index=False)
X = df.drop(columns=['alguna_vez_probation'])
y = df['alguna_vez_probation']
print(y.value_counts())

In [ ]:
cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

pre = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = ImbPipeline([
    ('pre', pre), ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced_subsample'))
])

xgb = ImbPipeline([
    ('pre', pre), ('smote', SMOTE(random_state=42)),
    ('model', XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.08, subsample=0.9, colsample_bytree=0.9, eval_metric='logloss', random_state=42, n_jobs=2))
])

rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)
p_rf = rf.predict(X_test)
p_xgb = xgb.predict(X_test)

In [ ]:
def m(y_true, y_pred):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'matriz_confusion': confusion_matrix(y_true, y_pred).tolist()
    }

mr = m(y_test, p_rf)
mx = m(y_test, p_xgb)
print('RandomForest:', mr)
print('XGBoost    :', mx)

best = 'XGBoost' if mx['f1'] >= mr['f1'] else 'RandomForest'
cm = confusion_matrix(y_test, p_xgb if best == 'XGBoost' else p_rf)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
plt.title(f'Matriz de Confusion - {best} (Probation)')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.tight_layout()
plt.savefig(fig_cm, dpi=150)
plt.show()

out = {'target': 'alguna_vez_probation', 'modelos': {'RandomForest': mr, 'XGBoost': mx}, 'mejor_modelo_f1': best}
metrics_path.write_text(json.dumps(out, indent=2), encoding='utf-8')
print('Metricas guardadas en:', metrics_path)